# Sequence-Aware Modeling Pipeline

This notebook builds a sequence-aware modeling pipeline on top of `data/sequencing_dataset.csv`.

It keeps the current wide checkpoint representation, engineers a compact second layer of causal sequence features, and compares three model families:

- class-weighted Logistic Regression
- XGBoost
- a small PyTorch MLP benchmark

The notebook reuses the frozen split artifacts already defined in the project.

## Modeling Note

An RNN is **not** the default choice here.

`sequencing_dataset.csv` is already a checkpoint-level table with aggregated sequence summaries rather than a raw event-by-event tensor. For this representation, boosted trees are usually the strongest first nonlinear baseline, while a small MLP is a reasonable neural benchmark. A true GRU/LSTM model would make more sense in a separate notebook built from the long event or possession sequences.

In [1]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / "data").exists())
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts" / "sequencing_model_pipeline"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(ARTIFACT_DIR / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "scored_after"
RUN_MLP = True


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, np.nan)
    return numerator.divide(denominator).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def entropy_from_shares(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    shares = frame[columns].clip(lower=0.0).astype(float)
    safe = shares.where(shares > 0.0, 1.0)
    return (-(shares * np.log(safe)).sum(axis=1)).fillna(0.0)


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["last15_sprint_share_of_cumul"] = safe_divide(out["last15_sprints"], out["cumul_sprints"])
    out["last15_hsr_share_of_cumul"] = safe_divide(out["last15_hsr"], out["cumul_hsr"])
    out["last15_shot_share_of_cumul"] = safe_divide(out["last15_shots"], out["cumul_shots"])
    out["last15_shot_on_target_rate"] = safe_divide(out["last15_shots_on_target"], out["last15_shots"])
    out["cumul_shot_on_target_rate"] = safe_divide(out["cumul_shots_on_target"], out["cumul_shots"])
    out["last15_under_pressure_shot_rate"] = safe_divide(out["last15_shots_under_press"], out["last15_shots"])
    out["cumul_under_pressure_shot_rate"] = safe_divide(out["cumul_shots_under_press"], out["cumul_shots"])
    out["last15_pass_passed_accuracy"] = safe_divide(out["last15_pass_passed_accurate"], out["last15_pass_passed"])
    out["last15_pass_received_accuracy"] = safe_divide(out["last15_pass_received_accurate"], out["last15_pass_received"])
    out["cumul_pass_passed_accuracy"] = safe_divide(out["cumul_pass_passed_accurate"], out["cumul_pass_passed"])
    out["cumul_pass_received_accuracy"] = safe_divide(out["cumul_pass_received_accurate"], out["cumul_pass_received"])

    out["seq15m_pressure_to_run_ratio"] = safe_divide(out["seq15m_pressure_total"], out["seq15m_run_total"])
    out["seq15m_pressure_to_shot_ratio"] = safe_divide(out["seq15m_pressure_total"], out["seq15m_shot_total"])
    out["seq15m_run_to_shot_ratio"] = safe_divide(out["seq15m_run_total"], out["seq15m_shot_total"])
    out["seq15m_forward_minus_backward_share"] = out["seq15m_forward_angle_share"] - out["seq15m_backward_angle_share"]
    out["seq15m_forward_minus_lateral_share"] = out["seq15m_forward_angle_share"] - out["seq15m_lateral_angle_share"]
    out["seq15m_recency_pressure_share"] = safe_divide(out["seq15m_recency_weighted_pressure_total"], out["seq15m_pressure_total"])
    out["seq15m_recency_run_share"] = safe_divide(out["seq15m_recency_weighted_run_total"], out["seq15m_run_total"])
    out["seq15m_recency_shot_share"] = safe_divide(out["seq15m_recency_weighted_shot_total"], out["seq15m_shot_total"])
    out["seq15m_activity_entropy"] = entropy_from_shares(
        out,
        ["seq15m_forward_angle_share", "seq15m_lateral_angle_share", "seq15m_backward_angle_share"],
    )

    out["seqpos_linked_pressure_per_possession"] = safe_divide(out["seqpos_linked_pressure_total"], out["seqpos_possession_count"])
    out["seqpos_transition_rate"] = safe_divide(out["seq15m_possession_transition_total"], out["seqpos_possession_count"])
    out["seqpos_run_volume_per_possession"] = safe_divide(out["seq15m_run_total"], out["seqpos_possession_count"])
    out["seqpos_shot_volume_per_possession"] = safe_divide(out["seq15m_shot_total"], out["seqpos_possession_count"])
    out["seqpos_matched_pressure_share"] = 1.0 - out["seqpos_unmatched_pressure_share"]

    out["seqextra_run_stage_top_minus_bottom"] = out["seqextra_run_stage_top_share"] - out["seqextra_run_stage_bottom_share"]
    out["seqextra_run_stage_entropy"] = entropy_from_shares(
        out,
        ["seqextra_run_stage_bottom_share", "seqextra_run_stage_middle_share", "seqextra_run_stage_top_share"],
    )
    out["seqextra_shot_right_minus_left_foot_share"] = (
        out["seqextra_shot_body_part_right_foot_share"] - out["seqextra_shot_body_part_left_foot_share"]
    )
    out["seqextra_non_regular_play_share"] = 1.0 - out["seqextra_shot_play_pattern_regular_play_share"]
    out["seqextra_set_piece_share"] = (
        out["seqextra_shot_play_pattern_corner_kick_share"]
        + out["seqextra_shot_play_pattern_indirect_free_kick_share"]
        + out["seqextra_shot_play_pattern_direct_free_kick_share"]
        + out["seqextra_shot_play_pattern_throw_in_share"]
        + out["seqextra_shot_play_pattern_penalty_share"]
    )
    out["seqextra_non_standard_technique_share"] = (
        out["seqextra_shot_technique_volley_share"]
        + out["seqextra_shot_technique_other_share"]
        + out["seqextra_shot_technique_overhead_kick_share"]
        + out["seqextra_shot_technique_lob_share"]
    )

    return out


def split_columns(df: pd.DataFrame, feature_columns: list[str]) -> tuple[list[str], list[str], list[str]]:
    numeric_cols: list[str] = []
    categorical_cols: list[str] = []
    boolean_cols: list[str] = []
    for column in feature_columns:
        dtype = df[column].dtype
        if pd.api.types.is_bool_dtype(dtype):
            boolean_cols.append(column)
        elif pd.api.types.is_numeric_dtype(dtype):
            numeric_cols.append(column)
        else:
            categorical_cols.append(column)
    return numeric_cols, categorical_cols, boolean_cols


def build_preprocessor(df: pd.DataFrame, feature_columns: list[str], scale_numeric: bool) -> ColumnTransformer:
    numeric_cols, categorical_cols, boolean_cols = split_columns(df, feature_columns)
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    transformers = []
    if numeric_cols:
        transformers.append(("num", Pipeline(steps=numeric_steps), numeric_cols))
    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            )
        )
    if boolean_cols:
        transformers.append(
            (
                "bool",
                Pipeline(
                    steps=[
                        (
                            "to_float",
                            FunctionTransformer(
                                lambda x: x.astype(float),
                                feature_names_out="one-to-one",
                            ),
                        ),
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                    ]
                ),
                boolean_cols,
            )
        )
    return ColumnTransformer(transformers=transformers)


def metric_row(y_true: pd.Series, y_score: np.ndarray) -> dict[str, float]:
    y_true = pd.Series(y_true).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if y_true.nunique() > 1 else np.nan,
        "brier_score": float(brier_score_loss(y_true, y_score)),
        "target_rate": float(y_true.mean()),
    }


def train_mlp(X_train: np.ndarray, y_train: np.ndarray, X_valid: np.ndarray, seed: int) -> np.ndarray:
    torch.manual_seed(seed)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32)

    model = torch.nn.Sequential(
        torch.nn.Linear(X_train.shape[1], 64),
        torch.nn.ReLU(),
        torch.nn.Dropout(0.2),
        torch.nn.Linear(64, 32),
        torch.nn.ReLU(),
        torch.nn.Dropout(0.1),
        torch.nn.Linear(32, 1),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    positive_count = max(float(y_train.sum()), 1.0)
    negative_count = max(float(len(y_train) - y_train.sum()), 1.0)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negative_count / positive_count], dtype=torch.float32))

    batch_size = min(128, len(X_train))
    for _ in range(120):
        permutation = torch.randperm(X_train_tensor.size(0))
        for start in range(0, X_train_tensor.size(0), batch_size):
            indices = permutation[start : start + batch_size]
            batch_x = X_train_tensor[indices]
            batch_y = y_train_tensor[indices]
            optimizer.zero_grad()
            loss = loss_fn(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        valid_logits = model(X_valid_tensor).squeeze(1)
        return torch.sigmoid(valid_logits).cpu().numpy()


/Users/jan/.matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /var/folders/pq/224wz8116dq05pr87bp6rkb80000gn/T/matplotlib-jclrovt4 because there was an issue with the default path (/Users/jan/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Matplotlib is building the font cache; this may take a moment.


In [2]:
sequence_df = pd.read_csv(DATA_DIR / "sequencing_dataset.csv")
row_folds = pd.read_csv(
    DATA_DIR / "splits" / "modeling_row_folds.csv",
    usecols=["fixture_id", "player_id", "checkpoint"],
)
row_folds["scored_after_eval_key"] = (
    row_folds["fixture_id"].astype(str)
    + "__"
    + row_folds["checkpoint"].astype(str)
    + "__"
    + row_folds["player_id"].astype(str)
)
fixture_assignments = pd.read_csv(DATA_DIR / "splits" / "fixture_assignments.csv")

sequence_df = sequence_df.merge(
    row_folds[["scored_after_eval_key", "fixture_id"]],
    on="scored_after_eval_key",
    how="left",
    validate="one_to_one",
)
sequence_df = sequence_df.merge(
    fixture_assignments,
    on="fixture_id",
    how="left",
    validate="many_to_one",
)

if sequence_df[["fixture_id", "fold", "partition_role"]].isna().any().any():
    raise ValueError("Split metadata join failed for sequencing_dataset.csv")

model_df = add_derived_features(sequence_df)

excluded_columns = {
    TARGET,
    "player_id",
    "jersey_number",
    "scored_after_eval_key",
    "fixture_id",
    "fold",
    "partition_role",
}

base_feature_columns = [
    column
    for column in model_df.columns
    if column not in excluded_columns and not column.startswith("seq")
]
sequence_feature_columns = [column for column in model_df.columns if column.startswith("seq") and "derived" not in column]
derived_feature_columns = sorted(set(model_df.columns) - set(sequence_df.columns))

feature_sets = {
    "base_context": list(dict.fromkeys(base_feature_columns)),
    "base_plus_sequence": list(dict.fromkeys(base_feature_columns + sequence_feature_columns)),
    "full_engineered": list(dict.fromkeys(base_feature_columns + sequence_feature_columns + derived_feature_columns)),
}

development_df = model_df[model_df["partition_role"] == "development"].copy().reset_index(drop=True)
fold_ids = sorted(development_df["fold"].dropna().unique())

print("Development rows:", len(development_df))
print("Target rate:", round(float(development_df[TARGET].mean()), 4))
print("Feature-set sizes:")
for name, columns in feature_sets.items():
    print(f"  {name}: {len(columns)} columns")


Development rows: 2830
Target rate: 0.0583
Feature-set sizes:
  base_context: 38 columns
  base_plus_sequence: 126 columns
  full_engineered: 126 columns


In [3]:
fold_rows: list[dict[str, object]] = []
summary_rows: list[dict[str, object]] = []
oof_predictions: list[pd.DataFrame] = []
interpretation_tables: dict[str, pd.DataFrame] = {}

for feature_set_name, feature_columns in feature_sets.items():
    for model_name in ["logreg", "xgb"]:
        fold_metrics = []
        current_oof = []
        first_fit_payload = None

        for fold_id in fold_ids:
            train_df = development_df[development_df["fold"] != fold_id].copy()
            valid_df = development_df[development_df["fold"] == fold_id].copy()

            if model_name == "logreg":
                pipeline = Pipeline(
                    steps=[
                        ("prep", build_preprocessor(train_df, feature_columns, scale_numeric=True)),
                        (
                            "model",
                            LogisticRegression(
                                class_weight="balanced",
                                max_iter=4000,
                                solver="liblinear",
                                random_state=RANDOM_STATE,
                            ),
                        ),
                    ]
                )
            else:
                positive_count = max(float(train_df[TARGET].sum()), 1.0)
                negative_count = max(float(len(train_df) - train_df[TARGET].sum()), 1.0)
                pipeline = Pipeline(
                    steps=[
                        ("prep", build_preprocessor(train_df, feature_columns, scale_numeric=False)),
                        (
                            "model",
                            XGBClassifier(
                                n_estimators=250,
                                max_depth=4,
                                learning_rate=0.05,
                                subsample=0.9,
                                colsample_bytree=0.9,
                                reg_alpha=0.0,
                                reg_lambda=1.0,
                                min_child_weight=3,
                                objective="binary:logistic",
                                eval_metric="logloss",
                                random_state=RANDOM_STATE,
                                scale_pos_weight=negative_count / positive_count,
                            ),
                        ),
                    ]
                )

            pipeline.fit(train_df[feature_columns], train_df[TARGET].astype(int))
            valid_scores = pipeline.predict_proba(valid_df[feature_columns])[:, 1]
            metric_values = metric_row(valid_df[TARGET], valid_scores)
            metric_values.update({"feature_set": feature_set_name, "model": model_name, "fold": fold_id})
            fold_metrics.append(metric_values)
            fold_rows.append(metric_values)

            current_oof.append(
                valid_df[["scored_after_eval_key", TARGET, "fold"]].assign(
                    feature_set=feature_set_name,
                    model=model_name,
                    prediction=valid_scores,
                )
            )

            if first_fit_payload is None:
                first_fit_payload = (pipeline, train_df.copy(), feature_columns.copy())

        fold_metric_df = pd.DataFrame(fold_metrics)
        summary_rows.append(
            {
                "feature_set": feature_set_name,
                "model": model_name,
                "mean_pr_auc": float(fold_metric_df["pr_auc"].mean()),
                "std_pr_auc": float(fold_metric_df["pr_auc"].std(ddof=0)),
                "mean_roc_auc": float(fold_metric_df["roc_auc"].mean()),
                "mean_brier_score": float(fold_metric_df["brier_score"].mean()),
            }
        )
        oof_predictions.append(pd.concat(current_oof, ignore_index=True))

        fitted_pipeline, fitted_train_df, fitted_columns = first_fit_payload
        prep = fitted_pipeline.named_steps["prep"]
        transformed_names = prep.get_feature_names_out()
        cleaned_names = [name.replace("num__", "").replace("cat__", "").replace("bool__", "") for name in transformed_names]
        if model_name == "logreg":
            coef_table = pd.DataFrame(
                {
                    "feature": cleaned_names,
                    "value": fitted_pipeline.named_steps["model"].coef_[0],
                }
            )
            coef_table["abs_value"] = coef_table["value"].abs()
            interpretation_tables[f"{feature_set_name}_{model_name}"] = coef_table.sort_values("abs_value", ascending=False)
        else:
            importance_table = pd.DataFrame(
                {
                    "feature": cleaned_names,
                    "value": fitted_pipeline.named_steps["model"].feature_importances_,
                }
            )
            interpretation_tables[f"{feature_set_name}_{model_name}"] = importance_table.sort_values("value", ascending=False)

if RUN_MLP:
    mlp_feature_columns = feature_sets["full_engineered"]
    mlp_fold_rows = []
    mlp_oof = []
    for fold_id in fold_ids:
        train_df = development_df[development_df["fold"] != fold_id].copy()
        valid_df = development_df[development_df["fold"] == fold_id].copy()
        prep = build_preprocessor(train_df, mlp_feature_columns, scale_numeric=True)
        X_train = prep.fit_transform(train_df[mlp_feature_columns])
        X_valid = prep.transform(valid_df[mlp_feature_columns])
        valid_scores = train_mlp(
            np.asarray(X_train, dtype=np.float32),
            train_df[TARGET].astype(int).to_numpy(dtype=np.float32),
            np.asarray(X_valid, dtype=np.float32),
            RANDOM_STATE,
        )
        metric_values = metric_row(valid_df[TARGET], valid_scores)
        metric_values.update({"feature_set": "full_engineered", "model": "mlp", "fold": fold_id})
        mlp_fold_rows.append(metric_values)
        fold_rows.append(metric_values)
        mlp_oof.append(
            valid_df[["scored_after_eval_key", TARGET, "fold"]].assign(
                feature_set="full_engineered",
                model="mlp",
                prediction=valid_scores,
            )
        )

    mlp_metric_df = pd.DataFrame(mlp_fold_rows)
    summary_rows.append(
        {
            "feature_set": "full_engineered",
            "model": "mlp",
            "mean_pr_auc": float(mlp_metric_df["pr_auc"].mean()),
            "std_pr_auc": float(mlp_metric_df["pr_auc"].std(ddof=0)),
            "mean_roc_auc": float(mlp_metric_df["roc_auc"].mean()),
            "mean_brier_score": float(mlp_metric_df["brier_score"].mean()),
        }
    )
    oof_predictions.append(pd.concat(mlp_oof, ignore_index=True))

fold_metrics_df = pd.DataFrame(fold_rows).sort_values(["model", "feature_set", "fold"]).reset_index(drop=True)
summary_df = pd.DataFrame(summary_rows).sort_values(["mean_pr_auc", "mean_roc_auc"], ascending=[False, False]).reset_index(drop=True)
oof_df = pd.concat(oof_predictions, ignore_index=True)

best_row = summary_df.iloc[0]
best_key = f"{best_row['feature_set']}_{best_row['model']}"
best_interpretation = interpretation_tables.get(best_key)

fold_metrics_df.to_csv(ARTIFACT_DIR / "fold_metrics.csv", index=False)
summary_df.to_csv(ARTIFACT_DIR / "model_summary.csv", index=False)
oof_df.to_csv(ARTIFACT_DIR / "oof_predictions.csv", index=False)
pd.DataFrame({"feature": derived_feature_columns}).to_csv(ARTIFACT_DIR / "derived_feature_columns.csv", index=False)

for key, table in interpretation_tables.items():
    table.head(50).to_csv(ARTIFACT_DIR / f"{key}_top_features.csv", index=False)

if best_interpretation is not None:
    plot_table = best_interpretation.head(20).iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(plot_table["feature"], plot_table["value"])
    plt.title(f"Top Features: {best_row['model']} on {best_row['feature_set']}")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "best_model_top_features.png", dpi=150)
    plt.close()

summary_df


,feature_set,model,mean_pr_auc,std_pr_auc,mean_roc_auc,mean_brier_score
0,base_context,xgb,0.116101,0.039697,0.642410,0.107340
1,base_context,logreg,0.097361,0.016022,0.618867,0.216323
2,base_plus_sequence,xgb,0.094217,0.021802,0.593141,0.087343
3,full_engineered,xgb,0.094217,0.021802,0.593141,0.087343
4,base_plus_sequence,logreg,0.082631,0.013064,0.575633,0.198316
5,full_engineered,logreg,0.082631,0.013064,0.575633,0.198316
6,full_engineered,mlp,0.077001,0.016181,0.520179,0.092239
